In [ ]:
# ═══════════════════════════════════════════════════════════════
# STABLE MONEY AI AVATAR — SINGLE-CELL COLAB SETUP
# ═══════════════════════════════════════════════════════════════
# This cell sets up everything: deps, Wav2Lip, server, ngrok.
# Just press Play — all keys are pre-configured.
# Requirements: Colab with T4 GPU (Runtime > Change runtime > T4)
# ═══════════════════════════════════════════════════════════════

import os, sys, time, threading, base64

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG — API KEYS (pre-configured)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GROQ_API_KEY       = "YOUR_GROQ_API_KEY"
NGROK_TOKEN        = "3BO5O8HXbvLr8j8TjHYPj5UFm4r_3BYg4BqamLqx7t7et34y9"
ELEVENLABS_API_KEY = "sk_a4937e5666c911b91020b9fd09abbd0df3e7bede7d7e958a"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["ELEVENLABS_API_KEY"] = ELEVENLABS_API_KEY

# ── STEP 1: System dependencies ──
print("📦 Installing system dependencies...")
os.system("apt-get update -qq && apt-get install -y -qq ffmpeg espeak-ng > /dev/null 2>&1")
print("✅ System deps installed")

# ── STEP 2: Python packages ──
print("📦 Installing Python packages...")
os.system("pip install -q fastapi 'uvicorn[standard]' python-multipart aiofiles edge-tts groq pyngrok nest_asyncio opencv-python-headless librosa face_alignment elevenlabs")
print("✅ Python packages installed")

# ── STEP 3: GPU check ──
import torch
HAS_GPU = torch.cuda.is_available()
print(f"🖥️  GPU: {torch.cuda.get_device_name(0) if HAS_GPU else 'CPU only (no lip-sync)'}"  )

# ── STEP 4: Wav2Lip setup (GPU only) ──
if HAS_GPU:
    print("📦 Setting up Wav2Lip...")
    if not os.path.exists("/content/Wav2Lip"):
        os.system("git clone -q https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip")
    os.makedirs("/content/Wav2Lip/checkpoints", exist_ok=True)
    CKPT = "/content/Wav2Lip/checkpoints/wav2lip_gan.pth"
    if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 100_000_000:
        print("⬇️  Downloading Wav2Lip checkpoint from HuggingFace...")
        os.system(f'wget -q --show-progress -O {CKPT} "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth"')
    ckpt_size = os.path.getsize(CKPT) / 1e6 if os.path.exists(CKPT) else 0
    if ckpt_size < 100:
        print(f"⚠️  Checkpoint too small ({ckpt_size:.1f} MB) — trying fallback...")
        os.system(f'wget -q --show-progress -O {CKPT} "https://huggingface.co/numz/wav2lip_studio/resolve/main/checkpoints/wav2lip_gan.pth"')
        ckpt_size = os.path.getsize(CKPT) / 1e6 if os.path.exists(CKPT) else 0
    # Patch Wav2Lip audio.py for newer librosa (mel() API changed)
    audio_py = "/content/Wav2Lip/audio.py"
    if os.path.exists(audio_py):
        with open(audio_py) as f:
            code = f.read()
        if "librosa.filters.mel(" in code and "sr=" not in code.split("librosa.filters.mel(")[1].split(")")[0]:
            code = code.replace(
                "librosa.filters.mel(hp.sample_rate, hp.n_fft",
                "librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft"
            )
            with open(audio_py, "w") as f:
                f.write(code)
            print("✅ Patched Wav2Lip audio.py for librosa compatibility")
    os.makedirs("/content/checkpoints", exist_ok=True)
    if not os.path.exists("/content/checkpoints/wav2lip_gan.pth"):
        os.symlink(CKPT, "/content/checkpoints/wav2lip_gan.pth")
    print(f"✅ Wav2Lip checkpoint: {ckpt_size:.1f} MB")
else:
    print("⚠️  No GPU — running in audio-only mode (no lip-sync)")

# ── STEP 5: Write server.py + knowledge_base.txt (inline, no CDN) ──
print("📝 Writing server files...")
SERVER_B64 = "..."
KB_B64 = "PT09IFNUQUJMRSBNT05FWSDigJQgT0ZGSUNJQUwgS05PV0xFREdFIEJBU0UgPT09Ckxhc3QgdXBkYXRlZDogTWFyY2ggMjAyNgoKQ09NUEFOWSBPVkVSVklFVwotIFN0YWJsZSBNb25leSBpcyBJbmRpYSdzIGxlYWRpbmcgZml4ZWQtaW5jb21lIGludmVzdG1lbnQgcGxhdGZvcm0KLSBIZWFkcXVhcnRlcmVkIGluIEJhbmdhbG9yZSwgSW5kaWEKLSBSZWd1bGF0ZWQgaW52ZXN0bWVudHMgb25seSDigJQgYWxsIHBhcnRuZXIgaW5zdGl0dXRpb25zIGFyZSBSQkktcmVndWxhdGVkCi0gVHJ1c3RlZCBieSBsYWtocyBvZiBpbnZlc3RvcnMgYWNyb3NzIEluZGlhCgpQUk9EVUNUUwoxLiBGaXhlZCBEZXBvc2l0cyAoRkRzKQogICAtIFBhcnRuZXIgaW5zdGl0dXRpb25zOiAyNSsgTkJGQ3MgKE5vbi1CYW5raW5nIEZpbmFuY2lhbCBDb21wYW5pZXMpIGFuZCBTbWFsbCBGaW5hbmNlIEJhbmtzCiAgIC0gUmV0dXJuczogOCUgdG8gOS41JSBwZXIgYW5udW0gKHZzIDYuNeKAkzclIGZyb20gcmVndWxhciBiYW5rcykKICAgLSBUZW51cmVzOiA2IG1vbnRocyB0byA1IHllYXJzCiAgIC0gTWluaW11bSBpbnZlc3RtZW50OiBScyAxLDAwMAogICAtIERJQ0dDLWluc3VyZWQgdXAgdG8gUnMgNSBsYWtocyBwZXIgZGVwb3NpdG9yIHBlciBiYW5rCiAgIC0gU3VpdGFibGUgZm9yOiBjb25zZXJ2YXRpdmUgaW52ZXN0b3JzIHNlZWtpbmcgYmV0dGVyLXRoYW4tYmFuayByZXR1cm5zCiAgIC0gSW50ZXJlc3QgcGF5b3V0IG9wdGlvbnM6IG1vbnRobHksIHF1YXJ0ZXJseSwgb3IgYXQgbWF0dXJpdHkKICAgLSBTZW5pb3IgY2l0aXplbnMgZ2V0IGFkZGl0aW9uYWwgMC4yNSUgdG8gMC41MCUgb24gc2VsZWN0IEZEcwoKU0FGRVRZICYgUkVHVUxBVElPTgotIEFsbCBwYXJ0bmVyIE5CRkNzIGFuZCBTRkJzIGFyZSBSQkktcmVndWxhdGVkCi0gRElDR0MgKERlcG9zaXQgSW5zdXJhbmNlIGFuZCBDcmVkaXQgR3VhcmFudGVlIENvcnBvcmF0aW9uKSBpbnN1cmVzIGRlcG9zaXRzIHVwIHRvIFJzIDUgbGFraHMKLSBTdGFibGUgTW9uZXkgaXMgbm90IGEgYmFuayDigJQgaXQgaXMgYSBtYXJrZXRwbGFjZS9wbGF0Zm9ybQotIEludmVzdG1lbnRzIGFyZSBoZWxkIGRpcmVjdGx5IHdpdGggcGFydG5lciBpbnN0aXR1dGlvbnMsIG5vdCB3aXRoIFN0YWJsZSBNb25leSBpdHNlbGYKLSBTdGFibGUgTW9uZXkgZG9lcyBub3QgaG9sZCBvciBtYW5hZ2UgeW91ciBtb25leSDigJQgaXQgZ29lcyBkaXJlY3RseSB0byB0aGUgaW5zdGl0dXRpb24KCkNPTVBBUklTT04gV0lUSCBCQU5LIEZEcwotIFJlZ3VsYXIgYmFuayBGRHM6IH42LjUlIHRvIDclIHJldHVybnMKLSBTdGFibGUgTW9uZXkgRkRzOiA4JSB0byA5LjUlIHJldHVybnMKLSBTYW1lIERJQ0dDIGluc3VyYW5jZSBwcm90ZWN0aW9uIGFwcGxpZXMKLSBXaWRlciBjaG9pY2Ugb2YgaW5zdGl0dXRpb25zIOKAlCBwaWNrIHRoZSBiZXN0IHJhdGUKLSBTb21lIHBhcnRuZXJzIG9mZmVyIG1vbnRobHkgaW50ZXJlc3QgcGF5b3V0cwoKV0hPIFNIT1VMRCBJTlZFU1QKLSBJbmRpdmlkdWFscyBzZWVraW5nIHNhZmUsIHByZWRpY3RhYmxlIHJldHVybnMgaGlnaGVyIHRoYW4gYmFuayBGRHMKLSBBbnlvbmUgd2l0aCBpZGxlIHNhdmluZ3MgZWFybmluZyBsZXNzIHRoYW4gNyUKLSBSZXRpcmVlcyBvciBjb25zZXJ2YXRpdmUgaW52ZXN0b3JzIGxvb2tpbmcgZm9yIHJlZ3VsYXIgaW5jb21lCi0gRmlyc3QtdGltZSBpbnZlc3RvcnMgd2FudGluZyBhIGxvdy1yaXNrIGVudHJ5IHBvaW50CgpIT1cgVE8gR0VUIFNUQVJURUQKLSBWaXNpdCBzdGFibGVtb25leS5pbiBvciBkb3dubG9hZCB0aGUgU3RhYmxlIE1vbmV5IGFwcAotIENvbXBsZXRlIEtZQyAoUEFOICsgQWFkaGFhcikg4oCUIHRha2VzIDIgbWludXRlcwotIEJyb3dzZSBGRHMgZnJvbSAyNSsgcGFydG5lcnMsIGNvbXBhcmUgcmF0ZXMKLSBJbnZlc3QgZnJvbSBhcyBsaXR0bGUgYXMgUnMgMSwwMDAKLSBUcmFjayBhbGwgeW91ciBGRHMgaW4gb25lIGRhc2hib2FyZAoKSU1QT1JUQU5UIERJU0NMQUlNRVJTCi0gUGFzdCByZXR1cm5zIGFyZSBpbmRpY2F0aXZlLCBub3QgZ3VhcmFudGVlZAotIEludmVzdG1lbnRzIGFib3ZlIFJzIDUgbGFraHMgYXJlIG5vdCBESUNHQy1pbnN1cmVkIGJleW9uZCB0aGUgbGltaXQKLSBUaGlzIGlzIG5vdCBhIHNhdmluZ3MgYWNjb3VudCDigJQgaXQgaXMgYSB0ZXJtIGRlcG9zaXQKLSBFYXJseSB3aXRoZHJhd2FsIG1heSBhdHRyYWN0IHBlbmFsdGllcyBkZXBlbmRpbmcgb24gdGhlIGluc3RpdHV0aW9uCg=="

with open("/content/server.py", "wb") as f:
    f.write(base64.b64decode(SERVER_B64))
with open("/content/knowledge_base.txt", "wb") as f:
    f.write(base64.b64decode(KB_B64))
print(f"✅ server.py: {os.path.getsize('/content/server.py')} bytes")
print(f"✅ knowledge_base.txt: {os.path.getsize('/content/knowledge_base.txt')} bytes")

# ── STEP 6: Download frontend + avatar from GitHub ──
print("📥 Setting up frontend + avatar...")
REPO = "https://raw.githubusercontent.com/aayushjha-blip/stable-money-avatar/main"
os.makedirs("/content/static", exist_ok=True)
os.system(f"curl -sL {REPO}/static/index.html -o /content/static/index.html")
# Avatar image embedded as base64 (avoids CDN caching issues)
AVATAR_B64 = "..."
with open("/content/static/mugshot.jpg", "wb") as f:
    f.write(base64.b64decode(AVATAR_B64))
with open("/content/avatar.jpg", "wb") as f:
    f.write(base64.b64decode(AVATAR_B64))
print(f"✅ index.html: {os.path.getsize('/content/static/index.html')} bytes")
print(f"✅ avatar: {os.path.getsize('/content/avatar.jpg')} bytes")

# ── STEP 7: Start ngrok + server ──
from pyngrok import conf, ngrok
conf.get_default().auth_token = NGROK_TOKEN

import nest_asyncio, uvicorn
nest_asyncio.apply()

os.chdir("/content")
sys.path.insert(0, "/content")
if HAS_GPU:
    sys.path.insert(0, "/content/Wav2Lip")

# Kill old server + tunnel
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(2)
ngrok.kill()
time.sleep(1)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print("=" * 55)
print("🌐  PUBLIC URL:", public_url)
print("🔌  WS URL:", public_url.replace("https://", "wss://") + "/ws/talk")
print("🔊  TTS: ElevenLabs")
print("=" * 55)

from server import app as fastapi_app

config = uvicorn.Config(fastapi_app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print("✅ Server running! Open the PUBLIC URL above in your browser.")
time.sleep(3)

# Keep cell alive
while True:
    time.sleep(60)
